In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold

In [2]:
df = pd.read_csv('../../data/features/rdkit_combined_df.csv')

In [3]:
df.columns

Index(['Unnamed: 0', 'MaxAbsEStateIndex', 'MaxEStateIndex',
       'MinAbsEStateIndex', 'MinEStateIndex', 'qed', 'SPS', 'MolWt',
       'HeavyAtomMolWt', 'ExactMolWt',
       ...
       'fr_sulfide', 'fr_sulfonamd', 'fr_sulfone', 'fr_term_acetylene',
       'fr_tetrazole', 'fr_thiazole', 'fr_thiocyan', 'fr_thiophene',
       'fr_unbrch_alkane', 'fr_urea'],
      dtype='object', length=211)

In [4]:
columns_to_drop = df.columns[:1].tolist()
X = df.drop(columns = columns_to_drop, axis=1)

In [5]:
X.shape

(1481, 210)

In [6]:
target_df = pd.read_csv('../../data/processed/Combined_df.csv')

In [7]:
target_df.columns

Index(['Unnamed: 0', 'SMILES', 'PCE'], dtype='object')

In [8]:
y = target_df['PCE']

In [9]:
y.shape

(1481,)

In [10]:
X.fillna(0, inplace = True)
X=X.select_dtypes(include='number')
X = X.astype(np.float32)
max_value = np.finfo(np.float32).max
X = X.loc[:, (X.iloc[:,:] <= max_value).all(axis=0)]

/home/hamza/anaconda3/envs/active_learning/lib/python3.12/site-packages/pandas/core/dtypes/astype.py:133: RuntimeWarning: overflow encountered in cast
  return arr.astype(dtype, copy=True)


In [11]:
X.shape

(1481, 205)

In [12]:
def remove_low_variance(input_data, threshold=0.1):
    selection = VarianceThreshold(threshold)
    selection.fit(input_data)
    return input_data[input_data.columns[selection.get_support(indices=True)]]

In [13]:
X = remove_low_variance(X, threshold=0.1)
X

,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,AvgIpc,...,fr_imide,fr_ketone,fr_ketone_Topliss,fr_methoxy,fr_nitrile,fr_piperdine,fr_pyridine,fr_sulfide,fr_thiazole,fr_thiophene
0,19.160440,19.160440,0.848696,-7.496612,36.548672,3170.614014,2938.773926,3167.584717,1182.0,1.475750,...,0.0,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,8.0
1,16.387346,16.387346,1.151380,-6.600159,33.325001,1103.494019,1030.917969,1102.484009,410.0,1.441446,...,0.0,2.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,2.0
2,16.532236,16.532236,0.526607,-5.374990,29.381819,1524.046021,1437.358032,1522.544678,550.0,1.450126,...,0.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
3,16.556942,16.556942,0.537908,-5.389678,29.000000,1556.043945,1469.355957,1554.534546,562.0,1.486817,...,0.0,2.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,4.0
4,10.857450,10.857450,0.565676,-6.364455,34.557377,883.375000,820.879028,882.385742,318.0,1.523381,...,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1476,16.281013,16.281013,0.672245,-7.047593,35.137615,1630.006958,1527.191040,1626.548340,572.0,1.476827,...,0.0,2.0,2.0,0.0,3.0,0.0,0.0,0.0,0.0,4.0
1477,15.752806,15.752806,0.631875,-5.663429,32.343750,1772.536011,1649.560059,1770.869629,658.0,1.459325,...,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,2.0,2.0
1478,15.117061,15.117061,0.143028,-5.309186,28.714285,1582.208008,1493.504028,1580.541626,562.0,1.460568,...,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,2.0,2.0
1479,15.893785,15.893785,1.172863,-6.997897,38.446602,1501.337036,1383.401001,1499.743042,554.0,1.503114,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0


In [14]:
X.shape

(1481, 126)

In [15]:
X.columns

Index(['MaxAbsEStateIndex', 'MaxEStateIndex', 'MinAbsEStateIndex',
       'MinEStateIndex', 'SPS', 'MolWt', 'HeavyAtomMolWt', 'ExactMolWt',
       'NumValenceElectrons', 'AvgIpc',
       ...
       'fr_imide', 'fr_ketone', 'fr_ketone_Topliss', 'fr_methoxy',
       'fr_nitrile', 'fr_piperdine', 'fr_pyridine', 'fr_sulfide',
       'fr_thiazole', 'fr_thiophene'],
      dtype='object', length=126)

In [16]:
def correlation(dataset, threshold):
    col_corr = set()  # Set of all the names of correlated columns
    corr_matrix = dataset.corr()
    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if (corr_matrix.iloc[i, j]) > threshold: # we are interested in absolute coeff value
                colname = corr_matrix.columns[i]  # getting the name of column
                col_corr.add(colname)
    return col_corr
corr_features = correlation(X, 0.8)
print('Features with correlation more than 0.8 :',len(set(corr_features)))
X = X.drop(list(corr_features & set(X.columns)), axis=1)

Features with correlation more than 0.8 : 61


In [17]:
X.shape

(1481, 65)

In [18]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [20]:
X_train.shape, X_test.shape

((1184, 65), (297, 65))

In [21]:
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

rf = RandomForestRegressor(random_state=42)

param_dist = {
    'n_estimators': [int(x) for x in np.linspace(start = 200, stop = 2000, num = 10)], # Number of trees in the forest
    'max_depth': [int(x) for x in np.linspace(10, 110, num = 11)], # Maximum depth of the tree
    'min_samples_split': [2, 5, 10],                   # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 4],                     # Minimum number of samples required to be at a leaf node
    'max_features': ['auto', 'sqrt', 'log2'],           # Number of features to consider at every split
    'bootstrap': [True, False]                         # Whether to use bootstrap samples when building trees
}

random_search = RandomizedSearchCV(estimator=rf, 
                                   param_distributions=param_dist, 
                                   n_iter=100,        # Number of iterations for searching the parameter space
                                   scoring='r2',  # Metric to optimize
                                   cv=5,              # 5-fold cross-validation
                                   verbose=2,         # Show progress
                                   random_state=42, 
                                   n_jobs=-1)         # Use all available cores

random_search.fit(X_train, y_train)

print("Best Parameters:", random_search.best_params_)

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END bootstrap=False, max_depth=110, max_features=sqrt, min_samples_leaf=2, min_samples_split=2, n_estimators=1200; total time=   6.8s
[CV] END bootstrap=False, max_depth=70, max_features=sqrt, min_samples_leaf=4, min_samples_split=5, n_estimators=400; total time=   1.9s
[CV] END bootstrap=False, max_depth=70, max_features=sqrt, min_samples_leaf=4, min_samples_split=5, n_estimators=400; total time=   1.9s
[CV] END bootstrap=True, max_depth=90, max_features=sqrt, min_samples_leaf=4, min_samples_split=5, n_estimators=1800; total time=   7.2s
[CV] END bootstrap=False, max_depth=20, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=1400; total time=   8.9s
[CV] END bootstrap=True, max_depth=80, max_features=log2, min_samples_leaf=2, min_samples_split=10, n_estimators=1600; total time=   6.1s
[CV] END bootstrap=True, max_depth=70, max_features=sqrt, min_samples_leaf=4, min_samples_split=2, n_estimators

/home/hamza/anaconda3/envs/active_learning/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:540: FitFailedWarning: 
120 fits failed out of a total of 500.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
95 fits failed with the following error:
Traceback (most recent call last):
  File "/home/hamza/anaconda3/envs/active_learning/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/hamza/anaconda3/envs/active_learning/lib/python3.12/site-packages/sklearn/base.py", line 1466, in wrapper
    estimator._validate_params()
  File "/home/hamza/anaconda3/envs/active_learning/lib/python3.12/site-packages/sklearn/base.py",

Best Parameters: {'n_estimators': 1800, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 100, 'bootstrap': False}


In [22]:
print("Best Parameters:", random_search.best_params_)

Best Parameters: {'n_estimators': 1800, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 100, 'bootstrap': False}
[CV] END bootstrap=False, max_depth=110, max_features=sqrt, min_samples_leaf=2, min_samples_split=2, n_estimators=1200; total time=   7.0s
[CV] END bootstrap=False, max_depth=70, max_features=sqrt, min_samples_leaf=4, min_samples_split=5, n_estimators=400; total time=   1.9s
[CV] END bootstrap=True, max_depth=80, max_features=log2, min_samples_leaf=1, min_samples_split=5, n_estimators=1600; total time=   6.6s
[CV] END bootstrap=True, max_depth=90, max_features=sqrt, min_samples_leaf=4, min_samples_split=5, n_estimators=1800; total time=   7.2s
[CV] END bootstrap=False, max_depth=70, max_features=log2, min_samples_leaf=4, min_samples_split=2, n_estimators=400; total time=   1.7s
[CV] END bootstrap=True, max_depth=80, max_features=log2, min_samples_leaf=2, min_samples_split=10, n_estimators=1600; total time=   6.1s
[CV] END bootstrap=False,

In [23]:
from sklearn.preprocessing import StandardScaler

In [24]:
# Scale the training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

In [26]:
# Step 2: Train the Gaussian Process Regression model
kernel = C(1.0, (1e-4, 1e1)) * RBF(length_scale=1.0, length_scale_bounds=(1e-4, 1e1))
gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10)
gpr.fit(X_train_scaled, y_train)

/home/hamza/anaconda3/envs/active_learning/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:659: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL_TERMINATION_IN_LNSRCH.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)


GaussianProcessRegressor(kernel=1**2 * RBF(length_scale=1),
                         n_restarts_optimizer=10)

In [ ]:


# Step 3: Active Learning with Random Forest Regression
n_iterations = 5  # Number of active learning iterations
rf_model = RandomForestRegressor(n_estimators=100)

for iteration in range(n_iterations):
    # Predict using GPR and calculate uncertainty
    y_pred, y_std = gpr.predict(X_full, return_std=True)
    
    # Step 4: Select samples based on uncertainty (lower uncertainty first)
    uncertainty = y_std.flatten()  # Get the standard deviation as uncertainty
    uncertain_samples = np.argsort(uncertainty)  # Indices of samples sorted by uncertainty (ascending)
    
    # Select the top N uncertain samples (you can adjust this)
    n_samples_to_add = 5
    new_samples_indices = uncertain_samples[:n_samples_to_add]
    
    # Add new samples to training data
    X_new = X_full[new_samples_indices]
    y_new = y_full[new_samples_indices]
    
    # Update training data
    X_train = np.vstack((X_train, X_new))
    y_train = np.hstack((y_train, y_new))
    
    # Step 5: Fit the Random Forest model
    rf_model.fit(X_train, y_train)

    # Print MAE for the current iteration
    rf_predictions = rf_model.predict(X_full)
    mae = mean_absolute_error(y_full, rf_predictions)
    print(f"Iteration {iteration + 1}, MAE: {mae:.3f}")

    # Optionally, retrain the GPR model after adding new samples
    gpr.fit(X_train, y_train)

# Visualize the results
plt.figure(figsize=(12, 6))
plt.plot(X_full, y_full, 'r:', label='True function')
plt.scatter(X_train, y_train, c='blue', s=50, label='Training data')
plt.scatter(X_full[new_samples_indices], y_new, c='green', s=100, edgecolor='black', label='Selected uncertain samples')
plt.plot(X_full, rf_model.predict(X_full), 'b-', label='Random Forest Prediction')
plt.fill_between(X_full.flatten(), 
                 rf_model.predict(X_full) - 1.96 * np.std(y_train), 
                 rf_model.predict(X_full) + 1.96 * np.std(y_train), 
                 color='gray', alpha=0.2, label='Uncertainty Interval (RF)')
plt.title('Active Learning with Gaussian Process and Random Forest')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.show()